# Oracle hindsight simulation (rebuttal)

Share of the oracle's sparsification gain recovered by a correct-by-construction uncertainty: the target is ONE draw y* per entry from each head's own predictive, the prediction is mu, u is the head's own sd. Same share definition as the paper's withholding table. Detected entries = y* > 0.

Precompute: `snake/scripts/canals/launch_oracle_sim_precompute.sh` (REAL=1 first for the equality check against canal 2).

In [3]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.append(str(PROJ))

RUN_ID      = "pr_classic3"
RESULTS     = PROJ / "results"
DATASETS    = ["10x_TuPro", "TLS", "LUNG_CANCER"]
RADIUS      = "r75"
LAMBDA_KEEP = "lambda_0.0"
MODELS      = ["heteroscedastic", "poisson_cp", "negativebinomial_hetero_noanchor_cp"]
MODEL_LABELS = {"heteroscedastic": "Heteroscedastic", "poisson_cp": "Poisson",
                "negativebinomial_hetero_noanchor_cp": "Negative binomial"}
DATASET_LABELS = {"10x_TuPro": "Tumor Profiler", "TLS": "Kidney/Lung TLS", "LUNG_CANCER": "NSCLC"}
# Table 4's four blocks: (case, protocol).
BLOCKS = [("lin_all", "gene_slide", "All entries, within (gene, slide)"),
          ("lin_all", "gene_slide_bin", "All entries, within mu bins"),
          ("lin_pos", "gene_slide", "Detected entries, within (gene, slide)"),
          ("lin_pos", "gene_slide_bin", "Detected entries, within mu bins")]


def run_dir(ds, m):
    d = RESULTS / ds / RUN_ID / f"_{ds}_hvg_top3000" / RADIUS / m
    return d / LAMBDA_KEEP if (d / LAMBDA_KEEP).is_dir() else d


def ause(curve):
    """As in the paper's withholding table."""
    c = np.asarray(curve, dtype=float)
    if c.size == 0 or not np.isfinite(c[0]) or c[0] == 0:
        return float("nan")
    return float(np.nanmean(c / c[0]))


def shares(sp, case, proto):
    """share_pred, share_mu_best (better of the two mu directions), AUSE of the oracle."""
    col = lambda s: f"curve_{s}_{proto}_{case}"
    if any(col(s) not in sp.columns for s in ("pred", "oracle", "mu_high", "mu_low")):
        return dict(u=np.nan, mu=np.nan, oracle=np.nan)
    o = ause(sp[col("oracle")])
    sh = lambda s: (1 - ause(sp[col(s)])) / (1 - o) if np.isfinite(o) and abs(1 - o) > 1e-9 else np.nan
    return dict(u=sh("pred"), mu=np.nanmax([sh("mu_high"), sh("mu_low")]), oracle=o)


rows, checks = [], []
for ds in DATASETS:
    for m in MODELS:
        d = run_dir(ds, m)
        real_p = d / "canal2_uncertainty" / "sparsification.parquet"
        real = pd.read_parquet(real_p) if real_p.is_file() else None
        eq_p = d / "oracle_sim" / "real" / "summary.json"
        eq = json.loads(eq_p.read_text()).get("equality_check") if eq_p.is_file() else None
        checks.append((ds, m, eq))
        seed_dirs = sorted((d / "oracle_sim").glob("seed*"))
        sims = [(pd.read_parquet(s / "sparsification.parquet"), json.loads((s / "summary.json").read_text()))
                for s in seed_dirs if (s / "sparsification.parquet").is_file()]
        for case, proto, _ in BLOCKS:
            r = shares(real, case, proto) if real is not None else dict(u=np.nan, mu=np.nan, oracle=np.nan)
            s = [shares(sp, case, proto) for sp, _ in sims]
            npos = [sm["n_positive_used"] for _, sm in sims]
            rows.append(dict(dataset=ds, model=m, case=case, protocol=proto, n_seeds=len(s),
                             real_u=r["u"], real_mu=r["mu"], real_oracle_ause=r["oracle"],
                             sim_u=np.nanmean([x["u"] for x in s]) if s else np.nan,
                             sim_u_sd=np.nanstd([x["u"] for x in s], ddof=1) if len(s) > 1 else np.nan,
                             sim_mu=np.nanmean([x["mu"] for x in s]) if s else np.nan,
                             sim_oracle_ause=np.nanmean([x["oracle"] for x in s]) if s else np.nan,
                             n_pos_real=sims[0][1]["n_positive_real"] if sims else np.nan,
                             n_pos_sim=np.mean(npos) if npos else np.nan))
T = pd.DataFrame(rows)

print("EQUALITY CHECK (oracle_sim --no-sim vs canal2_uncertainty, real y):")
for ds, m, eq in checks:
    if eq is None:
        print(f"  {ds:<12s} {m:<38s} not run")
    else:
        print(f"  {ds:<12s} {m:<38s} exact={eq.get('exact')}  max|diff|={eq.get('max_abs_diff')}  "
              f"NaN mismatches={len(eq.get('nan_pattern_mismatch', []))}")

print("\nSHARE OF THE ORACLE REDUCTION. real u = Table 4; sim u = correct-by-construction u on y* "
      "(mean +- SD over seeds); sim mu = predicted mean alone on y*, better direction.")
for case, proto, label in BLOCKS:
    print(f"\n--- {label} ---")
    print(f"  {'head':<20s}" + "".join(f"{DATASET_LABELS[ds]:>36s}" for ds in DATASETS))
    print(f"  {'':<20s}" + "".join(f"{'real u':>9s}{'sim u':>16s}{'sim mu':>11s}" for _ in DATASETS))
    for m in MODELS:
        line = f"  {MODEL_LABELS[m]:<20s}"
        for ds in DATASETS:
            r = T[(T.dataset == ds) & (T.model == m) & (T.case == case) & (T.protocol == proto)].iloc[0]
            sd = f"+-{r.sim_u_sd:.2f}" if np.isfinite(r.sim_u_sd) else ""
            line += f"{r.real_u:>+9.2f}{f'{r.sim_u:+.2f}{sd}':>16s}{r.sim_mu:>+11.2f}"
        print(line)

print("\nORACLE AUSE (lower = bigger oracle gain), real vs simulated, and detected-entry counts:")
print(T[["dataset", "model", "case", "protocol", "n_seeds", "real_oracle_ause", "sim_oracle_ause",
         "n_pos_real", "n_pos_sim"]].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

EQUALITY CHECK (oracle_sim --no-sim vs canal2_uncertainty, real y):
  10x_TuPro    heteroscedastic                        exact=True  max|diff|=0.0  NaN mismatches=0
  10x_TuPro    poisson_cp                             exact=True  max|diff|=0.0  NaN mismatches=0
  10x_TuPro    negativebinomial_hetero_noanchor_cp    exact=True  max|diff|=0.0  NaN mismatches=0
  TLS          heteroscedastic                        exact=False  max|diff|=8.857263138206672e-08  NaN mismatches=0
  TLS          poisson_cp                             exact=True  max|diff|=0.0  NaN mismatches=0
  TLS          negativebinomial_hetero_noanchor_cp    exact=False  max|diff|=2.6720563539583964e-08  NaN mismatches=0
  LUNG_CANCER  heteroscedastic                        exact=False  max|diff|=3.236890399049841e-07  NaN mismatches=0
  LUNG_CANCER  poisson_cp                             exact=False  max|diff|=4.6983965651392623e-07  NaN mismatches=0
  LUNG_CANCER  negativebinomial_hetero_noanchor_cp    exact=False  max

In [4]:
# Diagnostic: does y* keep the observed sparsity and expression level? Per gene,
# real y vs y*, from oracle_sim/seed*/diag_genes.parquet (per-gene arrays averaged over seeds).
# undetected = fraction of test spots with y <= 0; mean / var / mean over detected entries on the
# head's own scale (Gaussian log1p-CP10k, counts round(CP_T)). Decile = decile of the real gene mean.
N_DEC = 10
summ, dec_lines = [], []
for ds in DATASETS:
    for m in MODELS:
        files = sorted((run_dir(ds, m) / "oracle_sim").glob("seed*/diag_genes.parquet"))
        if not files:
            summ.append(dict(dataset=ds, model=m, n_seeds=0))
            continue
        D = [pd.read_parquet(f).set_index("gene") for f in files]
        G = sum(D) / len(D)
        dec = pd.qcut(G["mean_real"].rank(method="first"), N_DEC, labels=False)
        rho = lambda c: G[f"{c}_real"].corr(G[f"{c}_sim"], method="spearman")
        logratio = lambda c: float(np.nanmedian(np.log2(G[f"{c}_sim"] / G[f"{c}_real"])))
        summ.append(dict(dataset=ds, model=m, n_seeds=len(D),
                         undet_real=G["undetected_real"].mean(), undet_sim=G["undetected_sim"].mean(),
                         undet_rho=rho("undetected"),
                         undet_med_absdiff=float(np.median(np.abs(G["undetected_sim"] - G["undetected_real"]))),
                         mean_rho=rho("mean"), mean_log2ratio=logratio("mean"),
                         var_rho=rho("var"), var_log2ratio=logratio("var"),
                         meandet_rho=rho("mean_detected"), meandet_log2ratio=logratio("mean_detected")))
        u = G.groupby(dec)[["undetected_real", "undetected_sim"]].mean()
        dec_lines.append(f"  {DATASET_LABELS[ds]:<16s} {MODEL_LABELS[m]:<18s}"
                         + " ".join(f"{a:.2f}/{b:.2f}" for a, b in zip(u["undetected_real"], u["undetected_sim"])))

print("SPARSITY AND EXPRESSION LEVEL, real y vs simulated y*, per gene (rho = Spearman across genes;"
      "\nlog2ratio = median over genes of log2(sim / real), 0 = same level; undet = mean over genes of the"
      "\nundetected fraction):")
print(pd.DataFrame(summ).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print(f"\nUNDETECTED FRACTION by decile of the real gene mean (low -> high), real/sim:")
for line in dec_lines:
    print(line)

<env>/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)
<env>/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)
<env>/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


SPARSITY AND EXPRESSION LEVEL, real y vs simulated y*, per gene (rho = Spearman across genes;
log2ratio = median over genes of log2(sim / real), 0 = same level; undet = mean over genes of the
undetected fraction):
    dataset                               model  n_seeds  undet_real  undet_sim  undet_rho  undet_med_absdiff  mean_rho  mean_log2ratio  var_rho  var_log2ratio  meandet_rho  meandet_log2ratio
  10x_TuPro                     heteroscedastic        3       0.512      0.203      0.989              0.300     0.991           0.035    0.976         -0.242        0.862             -0.481
  10x_TuPro                          poisson_cp        3       0.520      0.431      0.991              0.100     0.989          -0.037    0.969         -1.860        0.760             -0.357
  10x_TuPro negativebinomial_hetero_noanchor_cp        3       0.520      0.506      0.993              0.030     0.989          -0.049    0.974         -0.245        0.916             -0.106
        TLS       